In [ ]:
from typing import Dict, List
from anarci import run_anarci
import torch
import torch.nn.functional as F

def extract_cdrs_with_positions(
    seq: str,
    maxlen: int,
    name: str = "input_seq",
    scheme: str = "imgt"
) -> Dict[str, object]:
    """
    extract_cdrs_with_positions
      res = (inputs, alignments, hits_meta, hits_table)

    return:
      {
        'name','scheme','chain_type','species',
        'cdrs': {
          'CDR1': {'seq': str, 'idx': List[int]},
          'CDR2': {'seq': str, 'idx': List[int]},
          'CDR3': {'seq': str, 'idx': List[int]},
        },
        'mask_cdr': List[bool],      # The CDR position（True）has the same length as the original input sequence.
        'mask_cdr01': List[int],     # 0/1
        'mask_cdr01_tensor': Tensor或None  # torch.int64；
      }
    """
    # run ANARCI
    res = run_anarci([(name, seq)], scheme=scheme, output=False,
                     assign_germline=True, allowed_species=None)

    # Retrieve the aligned number table; if the V zone is not recognized, issue a prompt.
    try:
        numbering = res[1][0][0][0]  # [ ((num, ins_char), aa), ... ]
    except Exception:
        raise RuntimeError(f"[{name}] No variable region detected (confirmed as AA sequence containing V region; or using scheme='imgt'）。")

    chain_type, species = "?", "?"
    try:
        meta = res[2][0][0]
        chain_type = meta.get("chain_type", "?")
        species    = meta.get("species", "?")
    except Exception:
        pass

    # establish an sequence index mapping.
    pos = []  # [{'num': int, 'ins': str|None, 'aa': str, 'raw_idx': int}]
    raw_idx = 0
    for (num, ins_char), aa in numbering:
        if aa == '-':      
            continue
        ins = None if ins_char == ' ' else ins_char
        pos.append({"num": num, "ins": ins, "aa": aa, "raw_idx": raw_idx})
        raw_idx += 1

    # IMGT 号段（H/L通用）
    ranges = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}

    # Extracting the CDR sequence and the original sequence index
    cdrs = {"CDR1": {"seq": "", "idx": []},
            "CDR2": {"seq": "", "idx": []},
            "CDR3": {"seq": "", "idx": []}}
    for p in pos:
        n = p["num"]
        for region, (lo, hi) in ranges.items():
            if lo <= n <= hi:
                cdrs[region]["seq"] += p["aa"]
                cdrs[region]["idx"].append(p["raw_idx"])

    #  CDR mask（bool & 0/1 ）
    mask_cdr = [False] * maxlen
    for r in ("CDR1", "CDR2", "CDR3"):
        for i in cdrs[r]["idx"]:
            if 0 <= i < len(mask_cdr):
                mask_cdr[i] = True
    mask_cdr01 = [1 if x else 0 for x in mask_cdr]

    # return PyTorch 0/1 tensor
    try:
        import torch
        mask_cdr01_tensor = torch.tensor(mask_cdr01, dtype=torch.long)
    except Exception:
        mask_cdr01_tensor = None

    return {
        "name": name,
        "scheme": scheme,
        "chain_type": chain_type,
        "species": species,
        "cdrs": cdrs,
        "mask_cdr": mask_cdr,                 # List[bool]
        "mask_cdr01": mask_cdr01,             # List[int]
        "mask_cdr01_tensor": mask_cdr01_tensor,  # torch.Tensor或None
    }

def expand_ones(signal: torch.Tensor, radius: int) -> torch.Tensor:
    """ signal expand +1 """
    kernel_size = 2 * radius + 1
    padding = (kernel_size - 1) // 2
    kernel = torch.ones(1, 1, kernel_size)
    x = signal.float().unsqueeze(0).unsqueeze(0)
    out = F.conv1d(x, kernel, padding=padding)
    return (out[0, 0] > 0).int()


if __name__ == "__main__":
    import json
    import torch
    import torch.nn.functional as F
    from tqdm import tqdm
    import os


    pdb_list = []
    for root, dirs, files in os.walk("/root/private_data/luog/AbAgKer/any_test/cdrs/"):
        for file in files:
            pdb_list.append(file.split(".")[0])

    ab_maxlen = 256
    res = {}
    with open("/root/private_data/luog/Data_AbAg/AbAgKer_all/json/AbAgI_9678.json", "r") as f:
        data = json.load(f)
        for i in tqdm(data):
            if i["pdb"] in pdb_list:
                continue
            try:
                H_cdr = extract_cdrs_with_positions(i["H"], maxlen=ab_maxlen, name=i["pdb"]+"_H", scheme="imgt")["mask_cdr01_tensor"]
            except Exception as e:
                print("H_cdr", e)
                H_cdr = torch.zeros( ab_maxlen)
            try:
                L_cdr = extract_cdrs_with_positions(i["L"], maxlen=ab_maxlen, name=i["pdb"]+"_L", scheme="imgt")["mask_cdr01_tensor"]
            except Exception as e:
                print("L_cdr", e)
                L_cdr = torch.zeros(ab_maxlen)


            cdrs = torch.cat([H_cdr, L_cdr], dim=0).long()
            # cdrs_pad = expand_ones(cdrs, 5).unsqueeze(0).long() # cdr expand version
            torch.save(cdrs, "/root/private_data/luog/AbAgKer/any_test/cdrs/"+i["pdb"]+".pt")

  0%|          | 45/9678 [00:08<27:19,  5.88it/s]

L_cdr [4KRL_IUMNM_L] 未识别到可变区（确认为AA序列且包含V区；或使用 scheme='imgt'）。
L_cdr [4KRO_VQHCN_L] 未识别到可变区（确认为AA序列且包含V区；或使用 scheme='imgt'）。


  0%|          | 46/9678 [00:08<26:15,  6.11it/s]

L_cdr [4KRP_UWICD_L] 未识别到可变区（确认为AA序列且包含V区；或使用 scheme='imgt'）。


 11%|█         | 1077/9678 [03:39<21:25,  6.69it/s] 

...

100%|█████████▉| 9670/9678 [29:09<00:01,  5.84it/s]

L_cdr [7JVB_HBUBU_L] 未识别到可变区（确认为AA序列且包含V区；或使用 scheme='imgt'）。


100%|█████████▉| 9673/9678 [29:10<00:00,  5.24it/s]

L_cdr [2X89_SKSQX_L] 未识别到可变区（确认为AA序列且包含V区；或使用 scheme='imgt'）。


100%|██████████| 9678/9678 [29:11<00:00,  5.53it/s]
